In [ ]:
# ------------------------------------------------------------------
# 🛑 THE NUCLEAR PURGE FIX FOR NUMPY 2.X CORRUPTION 🛑
# The previous scripts permanently contaminated your disk with Numpy 2.x files.
# We are going to physically delete them off the hard drive, then install
# the untouched 2024 version of WhisperX directly from an old Git commit.
# ------------------------------------------------------------------

# 1. Physically destroy all corrupted Numpy & Scipy files left over on the disk
!rm -rf /usr/local/lib/python*/dist-packages/numpy*
!rm -rf /usr/local/lib/python*/dist-packages/scipy*

# 2. Lock in the absolute stable bedrock of Python Data Science
!pip install -q numpy==1.26.4 scipy==1.13.1

# 3. Pull the exact stable 2024 WhisperX code (Before the Developer broke it)
!pip install -q pyannote.audio==3.1.1 pydub
!pip install -q git+https://github.com/m-bain/whisperx.git@v3.1.1

# 4. Patch the known HuggingFace Pipeline error
!sed -i 's/from transformers import Pipeline/from transformers.pipelines.base import Pipeline/g' /usr/local/lib/python*/dist-packages/whisperx/asr.py

# Create the input/output directories
import os
os.makedirs('/content/audios_to_transcribe', exist_ok=True)
os.makedirs('/content/outputs', exist_ok=True)

print("✅ Environment setup complete! Please upload your audio files to '/content/audios_to_transcribe' in the file explorer on the left.")

In [ ]:
import os
import whisperx
import gc
import torch
from google.colab import drive

# ----------------- MOUNT GOOGLE DRIVE -----------------
drive.mount('/content/drive')
# ------------------------------------------------------

# ----------------- CONFIGURATION -----------------
HF_TOKEN = "hf_YOUR_TOKEN_HERE" # <--- IMPORTANT: Replace with your HuggingFace token
MODEL_NAME = "large-v3" # or "large-v2"
COMPUTE_TYPE = "float16" # Optimal for L4 GPUs (utilizes Tensor Cores for maximum speed)
BATCH_SIZE = 64 # With an L4 (22GB VRAM), you can easily push batch size to 64 or even 128 for massive speedups!

# Use Google Drive paths directly:
INPUT_DIR = "/content/drive/MyDrive/audios_to_transcribe"
OUTPUT_DIR = "/content/drive/MyDrive/whisperx_outputs"

# Create directories if they don't exist yet on your Drive
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
# -------------------------------------------------

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    print("⚠️ WARNING: CPU mode detected. This will be very slow. Please enable a GPU!")
else:
    print(f"✅ Using device: {torch.cuda.get_device_name(0)}")

# Verify Pyannote Hugging Face access
if HF_TOKEN == "hf_YOUR_TOKEN_HERE":
    print("⚠️ WARNING: You have not set your HuggingFace token. Diarization will fail.")
    print("Go to https://hf.co/pyannote/speaker-diarization-3.1 to accept the user agreement, then put your token above.")

In [ ]:
import glob
from whisperx.utils import WriteVTT, WriteSRT, WriteJSON

# Find all audio files
audio_files = []
for ext in ('*.mp3', '*.wav', '*.m4a', '*.flac'):
    audio_files.extend(glob.glob(os.path.join(INPUT_DIR, ext)))

if not audio_files:
    print("❌ No audio files found. Please upload them to:", INPUT_DIR)
else:
    print(f"🔍 Found {len(audio_files)} audio files to process.")

# 1. Load the main Whisper model
print(f"Loading WhisperX {MODEL_NAME} model...")
model = whisperx.load_model(MODEL_NAME, device, compute_type=COMPUTE_TYPE)

# Initialize output writers
srt_writer = WriteSRT(OUTPUT_DIR)
vtt_writer = WriteVTT(OUTPUT_DIR)
json_writer = WriteJSON(OUTPUT_DIR)

for audio_path in audio_files:
    base_name = os.path.basename(audio_path)
    file_prefix = os.path.splitext(base_name)[0]
    print(f"\n🚀 Processing: {base_name}")

    # Load audio
    audio = whisperx.load_audio(audio_path)

    # ------------------------------------------------------------------
    # Step 1: Transcribe
    # ------------------------------------------------------------------
    print("  -> Transcribing...")
    result = model.transcribe(audio, batch_size=BATCH_SIZE)
    # Get audio language for alignment
    language = result["language"]

    # ------------------------------------------------------------------
    # Step 2: Align with Wav2Vec2
    # ------------------------------------------------------------------
    print("  -> Aligning words...")
    model_a, metadata = whisperx.load_align_model(language_code=language, device=device)
    result = whisperx.align(result["segments"], model_a, metadata, audio, device, return_char_alignments=False)

    # Free memory before diarization
    del model_a
    gc.collect()
    torch.cuda.empty_cache()

    # ------------------------------------------------------------------
    # Step 3: Diarization (Speaker identification)
    # ------------------------------------------------------------------
    if HF_TOKEN != "hf_YOUR_TOKEN_HERE" and not HF_TOKEN.startswith("hf_YOUR_TOKEN"):
        print("  -> Performing speaker diarization...")
        try:
            diarize_model = whisperx.DiarizationPipeline(use_auth_token=HF_TOKEN, device=device)

            # Force the model to recognize exactly 2 speakers for maximum accuracy:
            diarize_segments = diarize_model(audio, min_speakers=2, max_speakers=2)

            # Assign speakers to words
            result = whisperx.assign_word_speakers(diarize_segments, result)
        except Exception as e:
            print(f"  ❌ Diarization failed (Did you accept the Pyannote terms on HF?): {e}")

    # ------------------------------------------------------------------
    # Step 4: Save outputs
    # ------------------------------------------------------------------
    print("  -> Saving subtitle formats...")
    # Prepare arguments dictionary for the writers
    write_args = {"highlight_words": False, "max_line_width": None, "max_line_count": None}

    srt_writer(result, file_prefix, write_args)
    vtt_writer(result, file_prefix, write_args)
    json_writer(result, file_prefix, write_args)
    print(f"✅ Finished: {base_name}")

print("\n🎉 All files processed successfully!")